# 📘 Lesson 21: Support Vector Machines (SVM): Hard/Soft Margin & Kernels

Maximum Margin Hyperplane: Primal & Dual Lagrangian optimization, support vectors, soft-margin slack variables ($C$), and Kernel Trick (Linear, Polynomial, Radial Basis Function RBF).

---
**Interactive Step-by-Step Notebook**: Execute cells sequentially to observe data flow, mathematical transformations, and model evaluations.


### 🔹 Step 1: Implementation Block

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score
)


### 🔹 Load Dataset

**Objective**: Dataset Loading & Synthetic Data Generation.

- Ingests or synthesizes sample data. Inspects feature shapes, target distributions, and missing values.


In [ ]:
data=load_breast_cancer()

X=data.data #type:ignore
y=data.target #type:ignore

print("Feature shape", X.shape)
print("Target shape", X.shape)


### 🔹 Train-Test Split

**Objective**: Data Splitting & Feature Normalization.

- Splits data into independent train and test sets to evaluate generalization.

- Scales numerical features to zero mean ($\mu=0$) and unit variance ($\sigma=1$) to prevent features with large scales from dominating gradients or distance metrics.


In [ ]:
X_train, X_test, y_train, y_test=train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)

# Scaling
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)


### 🔹 Boosting (AdaBoost)

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada=AdaBoostClassifier(
    n_estimators=50 ,
    learning_rate=0.1,
    random_state=42
)

ada.fit(X_train_scaled, y_train)

train_pred=ada.predict(X_train_scaled)
test_pred=ada.predict(X_test_scaled)


### 🔹 Evaluation

**Objective**: Model Evaluation & Diagnostic Metrics.

- Calculates quantitative metrics ($R^2$, MSE, Accuracy, Precision, Recall, F1) to measure generalization performance.


In [ ]:
print("=== AdaBoost ===")

print("Train Accuracy:", accuracy_score(y_train, train_pred))

print("Test Accuracy:", accuracy_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))

print("Test F1:", f1_score(y_test, test_pred))


### 🔹 Gradient Boosting

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Experiment 1
gb_1=GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.1,
    random_state=42
)

gb_1.fit(X_train_scaled, y_train)

pred_1=gb_1.predict(X_test_scaled)

acc_1=accuracy_score(y_test, pred_1)
f1_1=f1_score(y_test, pred_1)


### 🔹 Experiment 2

**Objective**: Model Training & Parameter Optimization.

- Executes optimization (OLS normal equation, gradient descent, tree split search, or centroid convergence) on the training set.


In [ ]:
gb_2=GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42
)

gb_2.fit(X_train_scaled, y_train)

pred_2=gb_2.predict(X_test_scaled)

acc_2=accuracy_score(y_test, pred_2)
f1_2=f1_score(y_test, pred_2)


### 🔹 Experiment 3

**Objective**: Model Training & Parameter Optimization.

- Executes optimization (OLS normal equation, gradient descent, tree split search, or centroid convergence) on the training set.


In [ ]:
gb_3=GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.01,
    random_state=42
)

gb_3.fit(X_train_scaled, y_train)

pred_3=gb_3.predict(X_test_scaled)

acc_3=accuracy_score(y_test, pred_3)
f1_3=f1_score(y_test, pred_3)


### 🔹 Result Table

**Objective**: Execution & Utility Transformation.

- Transforms data features, computes intermediate statistics, or runs diagnostic evaluations.


In [ ]:
results=pd.DataFrame({
    "n_estimators":[50, 100, 200],
    "learning_rate":[0.1, 0.05, 0.01],
    "Accuracy":[acc_1, acc_2, acc_3],
    "F1-score": [f1_1, f1_2, f1_3]
})

print(results)


### 🔹 GridSearchCV

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
from sklearn.model_selection import GridSearchCV

gb_model=GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.05, 0.01],
    'max_depth': [1, 2, 3]
}

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)


### 🔹 Best Parameters

**Objective**: Execution & Utility Transformation.

- Transforms data features, computes intermediate statistics, or runs diagnostic evaluations.


In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

# Best Score
print("Best CV Score:")
print(grid_search.best_score_)


### 🔹 Best Model

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
best_model=grid_search.best_estimator_

# Evaluation
pred_grid=best_model.predict(X_test_scaled)

print("Best Model Test Accuracy:", accuracy_score(y_test, pred_grid))

print("Best Model F1 Score:", f1_score(y_test, pred_grid))


########## Stacking Classifier 

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import StackingClassifier


### 🔹 Base Models

**Objective**: Execution & Utility Transformation.

- Transforms data features, computes intermediate statistics, or runs diagnostic evaluations.


In [ ]:
base_models=[
    ("lr", LogisticRegression(max_iter=5000)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("knn", KNeighborsClassifier())
]


### 🔹 Meta learner

**Objective**: Model Training & Parameter Optimization.

- Executes optimization (OLS normal equation, gradient descent, tree split search, or centroid convergence) on the training set.


In [ ]:
meta_model=LogisticRegression(max_iter=5000)

# Build Stacking Model
stack_model=StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model
)

stack_model.fit(X_train_scaled, y_train)


### 🔹 Predictions

**Objective**: Model Inference & Prediction Generation.

- Computes predictions or predicted probability scores on unseen test samples.


In [ ]:
stack_pred=stack_model.predict(X_test_scaled)
stack_prob = stack_model.predict_proba(X_test_scaled)[:, 1] # type:ignore

# Evaluation
print("=== Stacking Classifier ===")

print("Accuracy:", accuracy_score(y_test, stack_pred))

print("ROC-AUC:", roc_auc_score(y_test, stack_prob))


## 🎯 Summary & Key Takeaways
1. **Core Insight**: Review the printed parameters, loss curves, and evaluation metrics above.
2. **Best Practice**: Always ensure proper feature scaling, validation splitting, and metric selection tailored to the problem distribution.
3. **Next Steps**: Compare these results with related models in subsequent lessons.
